<img src='http://www.tsc.uc3m.es/~navia/figures/logo_uc3m_foot.jpg' width=800 />

# Proyecto 3: Análisis de opinión

### Procesado del Lenguaje Natural

**Angel Navia Vázquez**

  * 1.1 (January 2026) Revised and updated version

Departamento de Teoría de la Señal y Comunicaciones

**Universidad Carlos III de Madrid**


En este notebook debéis entrenar un modelo para análisis de opinión:

- Para el entrenamiento del modelo se utilizarán los datos **data_project_NLP_3_train.csv**. En este fichero las opiniones positivas están codificadas como "1" y las negativas como "0".

- Tenéis total libertad para elegir la mejor implementación, tanto en lo que respecta a preprocesado, vectorización, como modelos de Machine Learning a utilizar.

- Incluid en el notebook todos los diferentes modelos evaluados, incluso las versiones preliminares, así como el modelo ganador final. Es decir, no borréis las pruebas intermedias que vayáis haciendo. Finalmente tendréis que elegir vuestro MODELO GANADOR, que será el que se evalúe sobre el  conjunto de test (**data_project_NLP_3_test.csv**).

- Presentar las prestaciones (AUC) del MEJOR MODELO sobre los datos de test (AUC).


In [ ]:
# =============================================================================
# Imports y configuración de optimización para Apple Silicon (M2 Pro)
# =============================================================================
import os
import multiprocessing as mp

# CPU multithread para BLAS, sklearn y PyTorch antes de importarlos
N_CPU = mp.cpu_count()                 # M2 Pro = 10–12 cores
N_THREADS = max(1, N_CPU - 1)
os.environ.setdefault("OMP_NUM_THREADS", str(N_THREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(N_THREADS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(N_THREADS))
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", str(N_THREADS))
os.environ.setdefault("NUMEXPR_NUM_THREADS", str(N_THREADS))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")
# Permite usar la CPU como respaldo en operaciones aún no implementadas en MPS
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

# NLP
import spacy
import nltk
from nltk.corpus import stopwords
try:
    from nltk.sentiment.vader import SentimentIntensityAnalyzer
except Exception:
    nltk.download('vader_lexicon', quiet=True)
    from nltk.sentiment.vader import SentimentIntensityAnalyzer

# Sklearn - preprocesado y vectorización
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler

# Sklearn - modelos
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.calibration import CalibratedClassifierCV

# Sklearn - evaluación
from sklearn.model_selection import (
    cross_val_score, StratifiedKFold, GridSearchCV, cross_val_predict, train_test_split
)
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from scipy.sparse import hstack, csr_matrix

# PyTorch - configurar threads y device (MPS = GPU integrada del M2 Pro)
import torch
torch.set_num_threads(N_THREADS)
try:
    torch.set_num_interop_threads(max(1, N_THREADS // 2))
except RuntimeError:
    pass  # ya configurado

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    USE_FP16 = False  # fp16 en MPS aún tiene operadores incompletos para entrenamiento
    print(f"PyTorch device: MPS (Apple Silicon GPU)")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    USE_FP16 = True
    print(f"PyTorch device: CUDA")
else:
    DEVICE = torch.device("cpu")
    USE_FP16 = False
    print(f"PyTorch device: CPU")

print(f"CPU cores: {N_CPU} | Threads usados: {N_THREADS}")

# Instalar transformers si no está disponible
try:
    import transformers
    print(f"transformers {transformers.__version__} ya disponible")
except ImportError:
    print("Instalando transformers...")
    import subprocess
    subprocess.check_call(['pip', 'install', 'transformers', 'accelerate'])
    import transformers
    print(f"transformers {transformers.__version__} instalado")

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, AutoModelForMaskedLM,
    DataCollatorForLanguageModeling, TrainingArguments, Trainer,
    EarlyStoppingCallback,
)
from torch.utils.data import Dataset

import time

# Descargar el modelo SpaCy si no está disponible
try:
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
except Exception:
    print("Descargando modelo SpaCy en_core_web_sm...")
    import subprocess
    subprocess.check_call(['python', '-m', 'spacy', 'download', 'en_core_web_sm'])
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

# Stopwords NLTK
try:
    stop_words = set(stopwords.words('english'))
except Exception:
    nltk.download('stopwords', quiet=True)
    stop_words = set(stopwords.words('english'))

print(f"\nEntorno listo. Device={DEVICE}, fp16={USE_FP16}")


In [2]:
# Cargar los datos de entrenamiento proporcionados
DATA_PATH = 'Proyecto_3_NLP_Datasets/'

df_train = pd.read_csv(DATA_PATH + 'data_project_NLP_3_train.csv')

print(f"Dimensiones del dataset de entrenamiento: {df_train.shape}")
print(f"\nDistribución de clases:")
print(df_train['opinion'].value_counts())
print(f"\nBalance: {df_train['opinion'].mean():.2%} positivas")
print(f"\nPrimeras filas:")
df_train.head(10)

Dimensiones del dataset de entrenamiento: (8000, 2)

Distribución de clases:
opinion
0    4122
1    3878
Name: count, dtype: int64

Balance: 48.48% positivas

Primeras filas:


,texto,opinion
0,@AmericanAir it's not a friend it's a legally ...,0
1,Ruukki 's order book at the end of 2010 was 30...,1
2,"""Gained 30 pounds while on prednisone. It was...",1
3,"""suffering from ME/CFS for over 25 yrs; morphi...",1
4,"""High Points:-Easy to read screen-Touchscreen ...",1
5,"""Day 1-Started with first dose after breakfast...",0
6,Apple's iPhone 6 Plus Amazingly Captures 41% o...,1
7,"""For about 6 years my physician prescribed Lip...",1
8,my poor mom can't sleep cause she had a bat in...,0
9,@LittleMissCindy I know it does!! But I really...,0


## Exploración y preprocesado de datos

In [3]:
# Exploración de los datos
print("Valores nulos:")
print(df_train.isnull().sum())
print(f"\nLongitud media de los textos: {df_train['texto'].str.len().mean():.0f} caracteres")
print(f"Longitud mediana: {df_train['texto'].str.len().median():.0f} caracteres")
print(f"Longitud máxima: {df_train['texto'].str.len().max()} caracteres")
print(f"Longitud mínima: {df_train['texto'].str.len().min()} caracteres")

# Rellenar posibles nulos
df_train['texto'] = df_train['texto'].fillna('')

# Separar features y target
X_train_raw = df_train['texto'].values
y_train = df_train['opinion'].values

print(f"\nTotal muestras: {len(X_train_raw)}")
print(f"Positivas (1): {sum(y_train==1)} ({sum(y_train==1)/len(y_train):.1%})")
print(f"Negativas (0): {sum(y_train==0)} ({sum(y_train==0)/len(y_train):.1%})")

Valores nulos:
texto      0
opinion    0
dtype: int64

Longitud media de los textos: 224 caracteres
Longitud mediana: 135 caracteres
Longitud máxima: 6182 caracteres
Longitud mínima: 61 caracteres

Total muestras: 8000
Positivas (1): 3878 (48.5%)
Negativas (0): 4122 (51.5%)


In [4]:
# Funciones de preprocesado

def clean_text(text):
    """Limpieza básica de texto"""
    # Eliminar HTML entities
    text = re.sub(r'&[a-zA-Z]+;', ' ', text)
    text = re.sub(r'&#\d+;', ' ', text)
    # Eliminar URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)
    # Eliminar menciones @usuario
    text = re.sub(r'@\w+', '', text)
    # Eliminar caracteres especiales pero mantener puntuación básica de sentimiento (!?)
    text = re.sub(r'[^a-zA-Z\s!?]', ' ', text)
    # Eliminar espacios múltiples
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

def preprocess_spacy(texts, batch_size=512):
    """Preprocesado con SpaCy: limpieza + lematización + eliminación de stopwords"""
    cleaned = [clean_text(t) for t in texts]
    processed = []
    for doc in nlp.pipe(cleaned, batch_size=batch_size):
        tokens = [token.lemma_ for token in doc 
                  if not token.is_stop and not token.is_punct and token.is_alpha and len(token) > 1]
        processed.append(' '.join(tokens))
    return processed

# Preprocesar datos de entrenamiento
print("Preprocesando textos con SpaCy (lematización + stopwords)...")
t0 = time.time()
X_train_processed = preprocess_spacy(X_train_raw)
print(f"Preprocesado completado en {time.time()-t0:.1f}s")

# También mantener versión solo con limpieza básica (sin lematización)
X_train_cleaned = [clean_text(t) for t in X_train_raw]

print(f"\nEjemplo original:  {X_train_raw[0][:100]}...")
print(f"Ejemplo limpio:    {X_train_cleaned[0][:100]}...")
print(f"Ejemplo procesado: {X_train_processed[0][:100]}...")

Preprocesando textos con SpaCy (lematización + stopwords)...
Preprocesado completado en 33.4s

Ejemplo original:  @AmericanAir it's not a friend it's a legally required chaperone on a school trip....
Ejemplo limpio:    it s not a friend it s a legally required chaperone on a school trip...
Ejemplo procesado: friend legally require chaperone school trip...


## Modelo 1: TF-IDF (word n-grams) + Logistic Regression (Baseline)

In [5]:
# =====================================================
# MODELO 1: TF-IDF (word) + Logistic Regression
# =====================================================
# Configuración de validación cruzada
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Vectorización TF-IDF con word n-grams sobre texto procesado (lematizado)
tfidf_word = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=50000,
    min_df=2,
    max_df=0.85,
    sublinear_tf=True,
    norm='l2'
)

X_tfidf_word = tfidf_word.fit_transform(X_train_processed)
print(f"TF-IDF word shape: {X_tfidf_word.shape}")

# Logistic Regression
lr_model = LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs', random_state=42)
scores_lr = cross_val_score(lr_model, X_tfidf_word, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
print(f"\nModelo 1 - LR + TF-IDF word (procesado):")
print(f"  AUC CV: {scores_lr.mean():.4f} (+/- {scores_lr.std():.4f})")

TF-IDF word shape: (8000, 20779)

Modelo 1 - LR + TF-IDF word (procesado):
  AUC CV: 0.8853 (+/- 0.0136)


## Modelo 2: TF-IDF (word + char n-grams) + Logistic Regression

In [6]:
# =====================================================
# MODELO 2: TF-IDF (word + char) + Logistic Regression
# =====================================================
# Usando texto RAW limpio (sin lematizar) - los char n-grams capturan morfología

tfidf_word_raw = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 2),
    max_features=50000,
    min_df=2,
    max_df=0.85,
    sublinear_tf=True
)

tfidf_char = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(2, 4),
    max_features=50000,
    min_df=2,
    max_df=0.85,
    sublinear_tf=True
)

from scipy.sparse import hstack

X_word_raw = tfidf_word_raw.fit_transform(X_train_cleaned)
X_char = tfidf_char.fit_transform(X_train_cleaned)
X_combined = hstack([X_word_raw, X_char])
print(f"TF-IDF word+char shape: {X_combined.shape}")

lr_model2 = LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs', random_state=42)
scores_lr2 = cross_val_score(lr_model2, X_combined, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
print(f"\nModelo 2 - LR + TF-IDF word+char (raw):")
print(f"  AUC CV: {scores_lr2.mean():.4f} (+/- {scores_lr2.std():.4f})")

TF-IDF word+char shape: (8000, 62402)

Modelo 2 - LR + TF-IDF word+char (raw):
  AUC CV: 0.9047 (+/- 0.0137)


## Modelo 3: TF-IDF + LinearSVC (calibrado)

In [7]:
# =====================================================
# MODELO 3: TF-IDF (word+char) + LinearSVC (calibrado para probabilidades)
# =====================================================
svc_model = CalibratedClassifierCV(
    LinearSVC(C=1.0, max_iter=5000, random_state=42),
    cv=5
)
scores_svc = cross_val_score(svc_model, X_combined, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
print(f"Modelo 3 - LinearSVC + TF-IDF word+char:")
print(f"  AUC CV: {scores_svc.mean():.4f} (+/- {scores_svc.std():.4f})")

Modelo 3 - LinearSVC + TF-IDF word+char:
  AUC CV: 0.9108 (+/- 0.0126)


## Modelo 4: TF-IDF + MultinomialNB

In [8]:
# =====================================================
# MODELO 4: TF-IDF + MultinomialNB
# =====================================================
nb_model = MultinomialNB(alpha=0.1)
scores_nb = cross_val_score(nb_model, X_combined, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
print(f"Modelo 4 - MultinomialNB + TF-IDF word+char:")
print(f"  AUC CV: {scores_nb.mean():.4f} (+/- {scores_nb.std():.4f})")

Modelo 4 - MultinomialNB + TF-IDF word+char:
  AUC CV: 0.8905 (+/- 0.0123)


## Modelo 5: TF-IDF + GradientBoosting

In [9]:
# =====================================================
# MODELO 5: TF-IDF + GradientBoosting (con reducción de dimensionalidad)
# =====================================================
# GradientBoosting es lento con matrices sparse muy grandes, usamos SVD para reducir
svd = TruncatedSVD(n_components=300, random_state=42)
X_svd = svd.fit_transform(X_combined)
print(f"SVD shape: {X_svd.shape}, varianza explicada: {svd.explained_variance_ratio_.sum():.2%}")

gb_model = GradientBoostingClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1, 
    subsample=0.8, random_state=42
)
scores_gb = cross_val_score(gb_model, X_svd, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
print(f"\nModelo 5 - GradientBoosting + TF-IDF+SVD:")
print(f"  AUC CV: {scores_gb.mean():.4f} (+/- {scores_gb.std():.4f})")

SVD shape: (8000, 300), varianza explicada: 31.47%

Modelo 5 - GradientBoosting + TF-IDF+SVD:
  AUC CV: 0.8666 (+/- 0.0244)


## Modelo 6: LR optimizado con GridSearchCV

In [10]:
# =====================================================
# MODELO 6: LR optimizado con GridSearchCV sobre TF-IDF word+char
# =====================================================
param_grid = {
    'C': [0.1, 0.5, 1.0, 5.0, 10.0],
    'solver': ['lbfgs', 'liblinear'],
}

grid_lr = GridSearchCV(
    LogisticRegression(max_iter=2000, random_state=42),
    param_grid,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0
)

print("Buscando mejores hiperparámetros para LR...")
grid_lr.fit(X_combined, y_train)
print(f"\nModelo 6 - LR optimizado + TF-IDF word+char:")
print(f"  Mejores parámetros: {grid_lr.best_params_}")
print(f"  AUC CV: {grid_lr.best_score_:.4f}")
scores_lr_opt = grid_lr.best_score_

Buscando mejores hiperparámetros para LR...

Modelo 6 - LR optimizado + TF-IDF word+char:
  Mejores parámetros: {'C': 10.0, 'solver': 'liblinear'}
  AUC CV: 0.9109


## Modelo 7: DistilBERT fine-tuned (Transformer)

In [12]:
# =====================================================
# MODELO 7: DistilBERT fine-tuned para clasificación de sentimiento
# =====================================================

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    auc = roc_auc_score(labels, probs)
    return {'auc': auc}

# Configuración
MODEL_NAME = 'distilbert-base-uncased'
MAX_LEN = 256

print(f"Cargando tokenizer y modelo: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Dividir en train/val para early stopping (90/10)
from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_raw, y_train, test_size=0.1, random_state=42, stratify=y_train
)

train_dataset = SentimentDataset(X_tr, y_tr, tokenizer, MAX_LEN)
val_dataset = SentimentDataset(X_val, y_val, tokenizer, MAX_LEN)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

# Cargar modelo preentrenado
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# Configuración del entrenamiento
training_args = TrainingArguments(
    output_dir='./distilbert_sentiment',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    weight_decay=0.01,
    learning_rate=2e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='auc',
    greater_is_better=True,
    logging_steps=50,
    fp16=False,
    report_to='none',
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

# Reemplazar NotebookProgressCallback por ProgressCallback para evitar RuntimeError
from transformers import ProgressCallback
from transformers.utils.notebook import NotebookProgressCallback
trainer.remove_callback(NotebookProgressCallback)
trainer.add_callback(ProgressCallback)

print(f"\nEntrenando DistilBERT en {device}...")
t0 = time.time()
trainer.train()
print(f"Entrenamiento completado en {(time.time()-t0)/60:.1f} minutos")

# Evaluar en validación
eval_results = trainer.evaluate()
print(f"\nModelo 7 - DistilBERT fine-tuned:")
print(f"  AUC Val: {eval_results['eval_auc']:.4f}")
distilbert_auc_val = eval_results['eval_auc']

Cargando tokenizer y modelo: distilbert-base-uncased
Train: 7200, Val: 800


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1472.43it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Entrenando DistilBERT en mps...


  4%|▎         | 50/1350 [00:24<09:19,  2.32it/s] 

{'loss': '0.691', 'grad_norm': '1.064', 'learning_rate': '9.8e-06', 'epoch': '0.1111'}


...

{'train_runtime': '716.5', 'train_samples_per_second': '30.15', 'train_steps_per_second': '1.884', 'train_loss': '0.2536', 'epoch': '3'}


There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].
100%|██████████| 1350/1350 [12:07<00:00,  1.86it/s]


Entrenamiento completado en 12.2 minutos


100%|██████████| 25/25 [00:07<00:00,  3.39it/s]


Modelo 7 - DistilBERT fine-tuned:
  AUC Val: 0.9603


## Modelo 7b: Re-entrenar DistilBERT con TODOS los datos de entrenamiento

In [13]:
# =====================================================
# MODELO 7b: Re-entrenar DistilBERT con TODOS los datos de train
# =====================================================
# Una vez confirmado que DistilBERT es el mejor modelo, re-entrenamos 
# usando el 100% de los datos de entrenamiento para maximizar el AUC en test

print("Re-entrenando DistilBERT con TODOS los datos de entrenamiento...")

full_train_dataset = SentimentDataset(X_train_raw, y_train, tokenizer, MAX_LEN)

# Recargar modelo limpio
model_full = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

training_args_full = TrainingArguments(
    output_dir='./distilbert_sentiment_full',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=50,
    fp16=False,
    report_to='none',
    save_strategy='epoch',
    seed=42,
)

trainer_full = Trainer(
    model=model_full,
    args=training_args_full,
    train_dataset=full_train_dataset,
)

# Reemplazar NotebookProgressCallback
trainer_full.remove_callback(NotebookProgressCallback)
trainer_full.add_callback(ProgressCallback)

t0 = time.time()
trainer_full.train()
print(f"Re-entrenamiento completado en {(time.time()-t0)/60:.1f} minutos")
print("Modelo ganador listo para evaluación en test.")

Re-entrenando DistilBERT con TODOS los datos de entrenamiento...


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1163.31it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
  3%|▎         | 50/1500 [00:24<10:27,  2.31it/s] 

{'loss': '0.6954', 'grad_norm': '1.765', 'learning_rate': '9.8e-06', 'epoch': '0.1'}


  7%|▋         | 100/1500 [00:45<09:47,  2.38it/s]

{'loss': '0.6207', 'grad_norm': '4.605', 'learning_rate': '1.98e-05', 'epoch': '0.2'}


...

{'loss': '0.15', 'grad_norm': '4.863', 'learning_rate': '7.286e-07', 'epoch': '2.9'}


100%|██████████| 1500/1500 [13:24<00:00,  1.77it/s]

{'loss': '0.08648', 'grad_norm': '0.6855', 'learning_rate': '1.429e-08', 'epoch': '3'}


100%|██████████| 1500/1500 [13:50<00:00,  1.81it/s]

{'train_runtime': '831', 'train_samples_per_second': '28.88', 'train_steps_per_second': '1.805', 'train_loss': '0.245', 'epoch': '3'}
Re-entrenamiento completado en 13.9 minutos
Modelo ganador listo para evaluación en test.


## Modelo 8 (NUEVO): Características de dominio (VADER + meta-features) + LR

Aprovechamos las técnicas vistas en los temas 6 (POS/NER), 7 (embeddings) y 12
(BERT) y añadimos **señales léxicas y meta-características** específicas para
sentiment-analysis que ayudan al ensemble final:

- Scores **VADER** (positivo, negativo, neutro y compuesto).
- Indicadores morfológicos: ratio de mayúsculas, número de `!` y `?`, presencia
  de negaciones, longitud y conteo de palabras.
- Polaridad de **TextBlob** (si está disponible).

Estas features se **concatenan** con el TF-IDF word+char y entrenamos un LR.


In [ ]:
# =====================================================
# MODELO 8: Características de dominio + TF-IDF word+char + LR
# =====================================================
sia = SentimentIntensityAnalyzer()

NEG_TOKENS = {"not", "n't", "no", "never", "nothing", "neither", "none", "nor"}

def _meta_features(text):
    if not isinstance(text, str):
        text = ""
    n_chars = len(text)
    n_words = max(1, len(text.split()))
    n_upper = sum(1 for c in text if c.isupper())
    n_excl = text.count('!')
    n_quest = text.count('?')
    has_neg = int(any(tok in text.lower().split() for tok in NEG_TOKENS))
    upper_ratio = n_upper / max(1, n_chars)
    vs = sia.polarity_scores(text)
    feats = [
        n_chars, n_words, upper_ratio, n_excl, n_quest, has_neg,
        vs['pos'], vs['neg'], vs['neu'], vs['compound'],
    ]
    return feats

# TextBlob es opcional
try:
    from textblob import TextBlob
    def _polarity(text):
        try:
            return TextBlob(text).sentiment.polarity
        except Exception:
            return 0.0
    HAS_TEXTBLOB = True
except Exception:
    HAS_TEXTBLOB = False

def build_meta_matrix(texts):
    feats = np.array([_meta_features(t) for t in texts], dtype=np.float32)
    if HAS_TEXTBLOB:
        pol = np.array([[_polarity(t)] for t in texts], dtype=np.float32)
        feats = np.hstack([feats, pol])
    return feats

print("Calculando meta-features...")
t0 = time.time()
META_train = build_meta_matrix(X_train_raw)
print(f"Meta features shape: {META_train.shape} en {time.time()-t0:.1f}s")

# Estandarizar las meta-features
scaler_meta = StandardScaler()
META_train_scaled = scaler_meta.fit_transform(META_train)

# Concatenar con TF-IDF word+char (X_combined ya disponible del Modelo 2)
X_meta_combined = hstack([X_combined, csr_matrix(META_train_scaled)]).tocsr()
print(f"TF-IDF + meta shape: {X_meta_combined.shape}")

lr_meta = LogisticRegression(C=1.0, max_iter=2000, solver='liblinear', random_state=42)
scores_lr_meta = cross_val_score(
    lr_meta, X_meta_combined, y_train, cv=cv, scoring='roc_auc', n_jobs=-1
)
print(f"\nModelo 8 - LR + TF-IDF word+char + meta-features:")
print(f"  AUC CV: {scores_lr_meta.mean():.4f} (+/- {scores_lr_meta.std():.4f})")


## Modelo 9 (NUEVO): RoBERTa pre-finetuned con DAPT + 5-fold CV ensemble

Aplicamos las mejoras del análisis:

1. **Modelo base**: `cardiffnlp/twitter-roberta-base-sentiment-latest` (ya
   pre-finetuneado en sentimiento, ideal para los tweets del dataset).
2. **MAX_LEN = 512** para no truncar las reseñas médicas largas.
3. **Domain-Adaptive Pretraining (DAPT)**: 1 epoch de Masked Language Modeling
   sobre el corpus de train antes del fine-tuning supervisado.
4. **5-fold StratifiedKFold ensemble**: entrenamos un modelo por fold y
   promediamos las probabilidades en test → reduce varianza del 90/10 split.
5. **Optimización Apple Silicon**: device MPS, threading multi-core, batch
   adaptado, scheduler `cosine`, early stopping.


In [ ]:
# =====================================================
# MODELO 9 - PASO 1: Domain-Adaptive Pretraining (DAPT) con MLM
# =====================================================
ROBERTA_MODEL = 'cardiffnlp/twitter-roberta-base-sentiment-latest'
MAX_LEN_BIG   = 512

print(f"DAPT: continual MLM pretraining de {ROBERTA_MODEL} sobre el corpus de train...")

tokenizer_big = AutoTokenizer.from_pretrained(ROBERTA_MODEL)

class MLMTextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len):
        self.enc = tokenizer(
            list(texts),
            truncation=True,
            padding='max_length',
            max_length=max_len,
            return_tensors='pt',
        )
    def __len__(self):
        return self.enc['input_ids'].size(0)
    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.enc.items()}

mlm_dataset = MLMTextDataset(X_train_raw, tokenizer_big, max_len=MAX_LEN_BIG)
mlm_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer_big, mlm=True, mlm_probability=0.15)

mlm_model = AutoModelForMaskedLM.from_pretrained(ROBERTA_MODEL)

DAPT_OUT = './roberta_dapt'
dapt_args = TrainingArguments(
    output_dir=DAPT_OUT,
    num_train_epochs=1,
    per_device_train_batch_size=8,        # 8 cabe en 16 GB unificada del M2 Pro
    gradient_accumulation_steps=2,        # batch efectivo 16
    learning_rate=5e-5,
    warmup_ratio=0.06,
    weight_decay=0.01,
    lr_scheduler_type='linear',
    logging_steps=100,
    save_strategy='no',
    fp16=USE_FP16,
    bf16=False,
    dataloader_num_workers=max(2, N_THREADS // 2),
    dataloader_pin_memory=False,          # innecesario en MPS
    report_to='none',
    seed=42,
    use_cpu=(DEVICE.type == 'cpu'),
)

dapt_trainer = Trainer(
    model=mlm_model,
    args=dapt_args,
    train_dataset=mlm_dataset,
    data_collator=mlm_collator,
)

t0 = time.time()
dapt_trainer.train()
mlm_model.save_pretrained(DAPT_OUT)
tokenizer_big.save_pretrained(DAPT_OUT)
print(f"DAPT terminado en {(time.time()-t0)/60:.1f} min. Modelo guardado en {DAPT_OUT}")

# Liberar memoria antes del siguiente paso
del mlm_model, dapt_trainer, mlm_dataset
import gc
gc.collect()
if DEVICE.type == 'mps':
    torch.mps.empty_cache()


In [ ]:
# =====================================================
# MODELO 9 - PASO 2: 5-fold CV fine-tune sobre el RoBERTa adaptado (DAPT)
# =====================================================
class SentimentDataset512(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN_BIG):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt',
        )
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels': torch.tensor(int(self.labels[idx]), dtype=torch.long),
        }

def compute_metrics_auc(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    return {'auc': roc_auc_score(labels, probs)}

N_FOLDS = 5
BATCH_TR  = 8
BATCH_EV  = 16
EPOCHS_FT = 3
LR_FT     = 2e-5

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
oof_roberta = np.zeros(len(y_train), dtype=np.float32)
fold_aucs = []
trainers_kfold = []  # guardamos los modelos para inferencia ensemble en test

t_global = time.time()
for fold, (idx_tr, idx_va) in enumerate(skf.split(X_train_raw, y_train), start=1):
    print(f"\n========= Fold {fold}/{N_FOLDS} =========")
    X_tr_f, X_va_f = X_train_raw[idx_tr], X_train_raw[idx_va]
    y_tr_f, y_va_f = y_train[idx_tr],     y_train[idx_va]

    # Cargamos los pesos del modelo DAPT (no los originales) -> mejor adaptación
    model_f = AutoModelForSequenceClassification.from_pretrained(
        DAPT_OUT, num_labels=2, ignore_mismatched_sizes=True
    )

    train_ds = SentimentDataset512(X_tr_f, y_tr_f, tokenizer_big, MAX_LEN_BIG)
    val_ds   = SentimentDataset512(X_va_f, y_va_f, tokenizer_big, MAX_LEN_BIG)

    args_f = TrainingArguments(
        output_dir=f'./roberta_fold{fold}',
        num_train_epochs=EPOCHS_FT,
        per_device_train_batch_size=BATCH_TR,
        per_device_eval_batch_size=BATCH_EV,
        gradient_accumulation_steps=2,    # batch efectivo 16
        learning_rate=LR_FT,
        warmup_ratio=0.1,
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='auc',
        greater_is_better=True,
        logging_steps=100,
        fp16=USE_FP16,
        bf16=False,
        dataloader_num_workers=max(2, N_THREADS // 2),
        dataloader_pin_memory=False,
        report_to='none',
        seed=42 + fold,
        save_total_limit=1,
        use_cpu=(DEVICE.type == 'cpu'),
    )
    trainer_f = Trainer(
        model=model_f,
        args=args_f,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics_auc,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
    )
    t0 = time.time()
    trainer_f.train()
    print(f"Fold {fold} entrenado en {(time.time()-t0)/60:.1f} min")

    # OOF predictions
    pred = trainer_f.predict(val_ds)
    probs_va = torch.softmax(torch.tensor(pred.predictions), dim=-1)[:, 1].numpy()
    oof_roberta[idx_va] = probs_va
    fold_auc = roc_auc_score(y_va_f, probs_va)
    fold_aucs.append(fold_auc)
    print(f"Fold {fold} AUC val = {fold_auc:.4f}")

    trainers_kfold.append(trainer_f)

    # Limpieza entre folds
    gc.collect()
    if DEVICE.type == 'mps':
        torch.mps.empty_cache()

distilbert_auc_val = float(np.mean(fold_aucs))   # mantener nombre por compat. con celda final
roberta_auc_kfold  = roc_auc_score(y_train, oof_roberta)

print(f"\nFolds completados en {(time.time()-t_global)/60:.1f} min")
print(f"AUC media por fold (val):  {np.mean(fold_aucs):.4f} (+/- {np.std(fold_aucs):.4f})")
print(f"AUC OOF global RoBERTa:    {roberta_auc_kfold:.4f}")


## Modelo 10 (NUEVO): Stacking — meta-learner sobre las probas OOF

Combinamos las **probabilidades Out-Of-Fold** del Transformer (RoBERTa+DAPT)
con las predicciones OOF de los modelos clásicos más fuertes (LR-TFIDF y
LR-meta), aprendiendo los pesos con un `LogisticRegression` (no overfit porque
las features son OOF).


In [ ]:
# =====================================================
# MODELO 10: Stacking - meta-learner sobre OOF probas
# =====================================================
print("Generando predicciones OOF de los modelos base clásicos...")

# Reconstruimos los modelos base con cross_val_predict (proba OOF)
oof_lr_word = cross_val_predict(
    LogisticRegression(C=1.0, max_iter=2000, solver='liblinear', random_state=42),
    X_combined, y_train, cv=cv, method='predict_proba', n_jobs=-1,
)[:, 1]

oof_lr_meta = cross_val_predict(
    LogisticRegression(C=1.0, max_iter=2000, solver='liblinear', random_state=42),
    X_meta_combined, y_train, cv=cv, method='predict_proba', n_jobs=-1,
)[:, 1]

# Stack de features OOF -> meta-learner
stack_train = np.column_stack([oof_roberta, oof_lr_word, oof_lr_meta])
print(f"Stack feature matrix: {stack_train.shape}")

print("\nAUC de cada base por separado (OOF):")
print(f"  RoBERTa+DAPT (5-fold): {roc_auc_score(y_train, oof_roberta):.4f}")
print(f"  LR + TFIDF word+char : {roc_auc_score(y_train, oof_lr_word):.4f}")
print(f"  LR + TFIDF + meta    : {roc_auc_score(y_train, oof_lr_meta):.4f}")

meta_lr = LogisticRegression(C=2.0, max_iter=2000, solver='liblinear', random_state=42)
scores_stack = cross_val_score(meta_lr, stack_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
print(f"\nModelo 10 - Stacking (meta-LR): AUC CV = {scores_stack.mean():.4f} "
      f"(+/- {scores_stack.std():.4f})")

# Entrenamos el meta-learner final con TODO
meta_lr.fit(stack_train, y_train)
print(f"Pesos del stacker: {dict(zip(['roberta','lr_word','lr_meta'], meta_lr.coef_[0]))}")
print(f"Intercept: {meta_lr.intercept_[0]:.4f}")


## Modelo 11 (NUEVO): Test-Time Augmentation (TTA)

En inferencia, generamos varias variantes de cada texto (texto original +
versión limpia + versión truncada) y promediamos las predicciones del
ensemble de RoBERTa. Mejora la robustez sin aumentar el coste de
entrenamiento.


In [ ]:
# =====================================================
# MODELO 11: Test-Time Augmentation helper
# =====================================================
def _tta_variants(text):
    """Genera 3 variantes simples de un texto para promediar predicciones."""
    if not isinstance(text, str):
        text = ""
    v1 = text
    v2 = clean_text(text)                              # versión limpia
    # versión truncada por palabras (útil para textos muy largos)
    words = text.split()
    v3 = " ".join(words[:200]) if len(words) > 200 else text
    return [v1, v2, v3]

def predict_roberta_ensemble(texts, trainers, tokenizer, max_len=MAX_LEN_BIG, use_tta=True):
    """Promedia probabilidades de los N folds entrenados, opcionalmente con TTA."""
    if use_tta:
        # Construimos un solo dataset con todas las variantes y mapeamos índices
        variants = [_tta_variants(t) for t in texts]
        flat = [v for sub in variants for v in sub]
        n_var = len(variants[0])
    else:
        flat = list(texts)
        n_var = 1

    flat_ds = SentimentDataset512(flat, [0]*len(flat), tokenizer, max_len)

    probs_sum = np.zeros(len(flat), dtype=np.float64)
    for trainer in trainers:
        out = trainer.predict(flat_ds)
        probs_sum += torch.softmax(torch.tensor(out.predictions), dim=-1)[:, 1].numpy()
    probs_sum /= len(trainers)

    # Promediar las variantes para cada texto original
    probs_sum = probs_sum.reshape(len(texts), n_var).mean(axis=1)
    return probs_sum

print("Helpers de TTA listos.")


## Comparación de todos los modelos y selección del MODELO GANADOR

In [ ]:
# =====================================================
# COMPARACIÓN DE TODOS LOS MODELOS
# =====================================================
print("=" * 75)
print("RESUMEN DE RESULTADOS - AUC en validación cruzada")
print("=" * 75)

resultados = {
    'Modelo 1: LR + TF-IDF word (procesado)':         scores_lr.mean(),
    'Modelo 2: LR + TF-IDF word+char (raw)':          scores_lr2.mean(),
    'Modelo 3: LinearSVC + TF-IDF word+char':         scores_svc.mean(),
    'Modelo 4: MultinomialNB + TF-IDF word+char':     scores_nb.mean(),
    'Modelo 5: GradientBoosting + TF-IDF+SVD':        scores_gb.mean(),
    'Modelo 6: LR optimizado + TF-IDF word+char':     scores_lr_opt,
    'Modelo 7: DistilBERT fine-tuned (val split)':    distilbert_auc_val,
    'Modelo 8: LR + TF-IDF + meta-features':          scores_lr_meta.mean(),
    'Modelo 9: RoBERTa+DAPT 5-fold ensemble (OOF)':   roberta_auc_kfold,
    'Modelo 10: Stacking meta-LR (RoBERTa+LR+meta)':  scores_stack.mean(),
}

resultados_sorted = dict(sorted(resultados.items(), key=lambda x: x[1], reverse=True))
for i, (nombre, auc) in enumerate(resultados_sorted.items(), 1):
    marker = " <-- GANADOR" if i == 1 else ""
    print(f"  {i:>2}. {nombre}: AUC = {auc:.4f}{marker}")

mejor_modelo = list(resultados_sorted.keys())[0]
mejor_auc = list(resultados_sorted.values())[0]
print(f"\n{'='*75}")
print(f"MODELO GANADOR: {mejor_modelo}")
print(f"AUC en validación: {mejor_auc:.4f}")
print(f"{'='*75}")
print("\nNota: para test usaremos el ENSEMBLE STACKING:")
print("  - 5 modelos RoBERTa (uno por fold) con TTA y promedio.")
print("  - LR + TF-IDF word+char entrenado en 100% del train.")
print("  - LR + TF-IDF + meta-features entrenado en 100% del train.")
print("  - Combinados por el meta-LR aprendido sobre OOF.")


## Evaluación del MODELO GANADOR en el conjunto de test

In [ ]:
# =====================================================
# Evaluación del MODELO GANADOR (Stacking + TTA) sobre el conjunto de test
# =====================================================
import time
t_ini = time.time()

# Cargar los datos de test
df_test = pd.read_csv(DATA_PATH + 'data_project_NLP_3_test.csv')
df_test['texto'] = df_test['texto'].fillna('')
X_test_raw = df_test['texto'].values
y_test = df_test['opinion'].values

print(f"Datos de test cargados: {len(X_test_raw)} muestras")
print(f"Distribución: {sum(y_test==1)} positivas, {sum(y_test==0)} negativas")

# --- 1) Predicciones de RoBERTa (5 folds) con TTA ---
print("\n[1/4] Inferencia ensemble RoBERTa+DAPT con TTA...")
prob_roberta_test = predict_roberta_ensemble(
    X_test_raw, trainers_kfold, tokenizer_big, MAX_LEN_BIG, use_tta=True
)

# --- 2) Predicciones LR + TF-IDF word+char (re-entrenado con 100% del train) ---
print("[2/4] Modelo clásico LR + TF-IDF word+char...")
X_test_cleaned = [clean_text(t) for t in X_test_raw]
X_test_word    = tfidf_word_raw.transform(X_test_cleaned)
X_test_char    = tfidf_char.transform(X_test_cleaned)
X_test_combined = hstack([X_test_word, X_test_char])

lr_word_full = LogisticRegression(C=1.0, max_iter=2000, solver='liblinear', random_state=42)
lr_word_full.fit(X_combined, y_train)
prob_lr_word_test = lr_word_full.predict_proba(X_test_combined)[:, 1]

# --- 3) Predicciones LR + TF-IDF + meta-features ---
print("[3/4] Modelo clásico LR + TF-IDF + meta-features...")
META_test = build_meta_matrix(X_test_raw)
META_test_scaled = scaler_meta.transform(META_test)
X_test_meta_combined = hstack([X_test_combined, csr_matrix(META_test_scaled)]).tocsr()

lr_meta_full = LogisticRegression(C=1.0, max_iter=2000, solver='liblinear', random_state=42)
lr_meta_full.fit(X_meta_combined, y_train)
prob_lr_meta_test = lr_meta_full.predict_proba(X_test_meta_combined)[:, 1]

# --- 4) Stacking meta-learner ---
print("[4/4] Stacking meta-learner...")
stack_test = np.column_stack([prob_roberta_test, prob_lr_word_test, prob_lr_meta_test])
preds_test = meta_lr.predict_proba(stack_test)[:, 1]

# AUC componente a componente y final
print("\n--- AUC por componente en TEST ---")
print(f"  RoBERTa+DAPT 5-fold ensemble + TTA:  {roc_auc_score(y_test, prob_roberta_test):.4f}")
print(f"  LR + TF-IDF word+char (full train):  {roc_auc_score(y_test, prob_lr_word_test):.4f}")
print(f"  LR + TF-IDF + meta-features:         {roc_auc_score(y_test, prob_lr_meta_test):.4f}")

AUC_test = roc_auc_score(y_test, preds_test)
print(f'\n{"="*75}')
print(f'AUC del ensemble final en test = {AUC_test}')
print(f'{"="*75}')
print("Tiempo de análisis en minutos = {:.2f}".format((time.time() - t_ini)/60))
